# `nn.GRU` from scratch

A **GRU** (Gated Recurrent Unit) is a recurrent network that processes a sequence one step at a time, maintaining a **hidden state** $h_t$ that summarizes everything seen so far. Two **gates** (reset and update) control how much of the past to keep and how much of the new input to write — this lets it model long sequences without the vanishing-gradient issues of a plain RNN.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

## 1. The gates

At each timestep $t$, given the input $x_t \in \mathbb{R}^{I}$ and the previous hidden state $h_{t-1} \in \mathbb{R}^{H}$, the GRU computes three quantities :

**Reset gate** — how much of the past hidden state to forget when proposing a new candidate :
$$r_t = \sigma\big(W_{ir} x_t + b_{ir} + W_{hr} h_{t-1} + b_{hr}\big)$$

**Update gate** — how much to keep the old state vs. write the new candidate :
$$z_t = \sigma\big(W_{iz} x_t + b_{iz} + W_{hz} h_{t-1} + b_{hz}\big)$$

**Candidate state** — the new content, where the reset gate gates the hidden contribution :
$$n_t = \tanh\big(W_{in} x_t + b_{in} + r_t \odot (W_{hn} h_{t-1} + b_{hn})\big)$$

**New hidden state** — a convex mix controlled by the update gate :
$$h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}$$

$\sigma$ is the sigmoid, $\odot$ is element-wise product. If $z_t \to 1$ the unit **keeps** its old state (good for long-term memory) ; if $z_t \to 0$ it **overwrites** with the new candidate.

## 2. Parameters

PyTorch packs the three gates' weights into single matrices (the factor 3 = reset, update, candidate) :

| Tensor | Shape | Meaning |
|---|---|---|
| `weight_ih` | $(3H,\ I)$ | input→gates : $W_{ir}, W_{iz}, W_{in}$ stacked |
| `weight_hh` | $(3H,\ H)$ | hidden→gates : $W_{hr}, W_{hz}, W_{hn}$ stacked |
| `bias_ih` | $(3H,)$ | $b_{ir}, b_{iz}, b_{in}$ |
| `bias_hh` | $(3H,)$ | $b_{hr}, b_{hz}, b_{hn}$ |

Total for one layer : $3H \cdot I + 3H \cdot H + 3H + 3H = 3H(I + H + 2)$. Like a conv, this is **independent of the sequence length** — the same weights are reused at every timestep (weight sharing in time).

## 3. From-scratch implementation

We reuse PyTorch's packed weights and split them into the three gates with `chunk(3)`. Note that `weight_ih` is $(3H, I)$, so $x_t W_{ih}^T$ gives $(B, 3H)$.

In [ ]:
def gru_step_scratch(x_t, h_prev, w_ih, w_hh, b_ih, b_hh):
    # x_t: (B, I), h_prev: (B, H)
    gi = x_t   @ w_ih.T + b_ih      # (B, 3H)  input contribution to the 3 gates
    gh = h_prev @ w_hh.T + b_hh     # (B, 3H)  hidden contribution to the 3 gates

    i_r, i_z, i_n = gi.chunk(3, dim=1)
    h_r, h_z, h_n = gh.chunk(3, dim=1)

    r = torch.sigmoid(i_r + h_r)               # reset gate
    z = torch.sigmoid(i_z + h_z)               # update gate
    n = torch.tanh(i_n + r * h_n)              # candidate (reset gates the hidden part)
    h = (1 - z) * n + z * h_prev               # new hidden state
    return h


def gru_scratch(x, h0, gru_module):
    # x : (B, T, I)  — run the from-scratch GRU over the whole sequence
    w_ih = gru_module.weight_ih_l0
    w_hh = gru_module.weight_hh_l0
    b_ih = gru_module.bias_ih_l0
    b_hh = gru_module.bias_hh_l0

    B, T, I = x.shape
    h = h0.squeeze(0)                          # (B, H)
    outputs = []
    for t in range(T):
        h = gru_step_scratch(x[:, t, :], h, w_ih, w_hh, b_ih, b_hh)
        outputs.append(h)
    out = torch.stack(outputs, dim=1)          # (B, T, H)
    return out, h.unsqueeze(0)

## 4. Verify against `nn.GRU`

In [ ]:
I, H = 5, 8
gru_module = nn.GRU(input_size=I, hidden_size=H, num_layers=1, batch_first=True)

B, T = 4, 10
x  = torch.randn(B, T, I)
h0 = torch.zeros(1, B, H)

out_torch, hT_torch     = gru_module(x, h0)
out_scratch, hT_scratch = gru_scratch(x, h0, gru_module)

print("output torch   :", tuple(out_torch.shape))
print("output scratch :", tuple(out_scratch.shape))
print("max abs diff (sequence) :", (out_torch - out_scratch).abs().max().item())
print("max abs diff (final h)  :", (hT_torch - hT_scratch).abs().max().item())
print("match :", torch.allclose(out_torch, out_scratch, atol=1e-5))

In [ ]:
# parameter count
gru_module = nn.GRU(5, 8, num_layers=1)
params    = sum(p.numel() for p in gru_module.parameters())
params_th = 3 * 8 * (5 + 8 + 2)     # 3H(I + H + 2)
print(f"empirical : {params}")
print(f"theory    : {params_th}")
print(f"match     : {params == params_th}")

## 5. Stacking and direction

**Multi-layer** (`num_layers=L`) : the output sequence of layer $\ell$ becomes the input sequence of layer $\ell+1$. Layer 0 has input size $I$ ; layers $\geq 1$ have input size $H$ (the hidden size of the layer below). Total params $\approx 3H(I+H+2) + (L-1)\cdot 3H(2H+2)$.

**Bidirectional** (`bidirectional=True`) : runs one GRU left→right and another right→left, then concatenates → output dim $2H$. Doubles the parameters. Used when the whole sequence is available (not for causal/autoregressive tasks).

**Returns** : `nn.GRU` returns `(output, h_n)` where
- `output` : $(B, T, H)$ — the hidden state at **every** timestep
- `h_n` : $(L, B, H)$ — the **final** hidden state of each layer

In CPC we use `h_n[-1]` (final hidden of the last layer) as the context vector $c_t$ — it summarizes the whole context sequence causally.